# QuadWitch Pitch EDA: does the quarterly rebalance close distort SPY, and does it reverse?

**The mechanism.** On the third Friday of March/June/September/December, S&P quarterly share and float updates, index
futures expiry and stock/index option expiry all hit the closing auction. If the forced trading pushes the close,
it should reverse overnight and the next day.

**Data.** SPY daily open/close (`Overnight Pitch/data/panel.csv`), 2004–2026. Quad-witching Fridays and monthly opex
Fridays vs other Fridays: the day's return, close → next open, and the next day. 2004–14 vs 2015–.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from scipy import stats
p=pd.read_csv("../Overnight Pitch/data/panel.csv",parse_dates=["date"])
s=p[p.ticker=="SPY"].set_index("date").sort_index()
f=s.adj_close/s.close
on=np.log(s.open.shift(-1)*f.shift(-1)/(s.close*f))*1e4          # close -> next open (dividend-adjusted)
day=np.log(s.adj_close/s.adj_close.shift(1))*1e4
cal=s.index
def third_fri(y,m):
    d=pd.date_range(pd.Timestamp(y,m,1),periods=31,freq="D"); fr=d[(d.weekday==4)&(d.month==m)][2]
    return cal[cal.searchsorted(fr,side="right")-1]
qw={third_fri(y,m) for y in range(2004,2027) for m in (3,6,9,12) if pd.Timestamp(y,m,28)<=cal[-1]}
mo={third_fri(y,m) for y in range(2004,2027) for m in range(1,13) if m not in (3,6,9,12) and pd.Timestamp(y,m,28)<=cal[-1]}
D=pd.DataFrame({"day":day,"next_on":on,"next_day":day.shift(-1)})
D["kind"]=np.where(D.index.isin(qw),"quad witching (index rebalance)",np.where(D.index.isin(mo),"monthly opex Friday",np.where(D.index.dayofweek==4,"other Friday","other")))
for per,m in [("2004-2014",D.index<"2015"),("2015-",D.index>="2015")]:
    b=D[m&(D.kind=="other Friday")]
    for k in ["quad witching (index rebalance)","monthly opex Friday"]:
        a=D[m&(D.kind==k)]
        print(per,f"{k:32s} n {len(a)}: "+" | ".join(f"{c} {a[c].mean():+.1f} vs {b[c].mean():+.1f} (t {stats.ttest_ind(a[c].dropna(),b[c].dropna(),equal_var=False)[0]:+.2f})" for c in ["day","next_on","next_day"]))

2004-2014 quad witching (index rebalance)  n 44: day +10.5 vs -0.4 (t +0.74) | next_on +8.0 vs +3.5 (t +0.40) | next_day -0.8 vs -1.7 (t +0.04)
2004-2014 monthly opex Friday              n 88: day +7.6 vs -0.4 (t +0.66) | next_on +1.3 vs +3.5 (t -0.22) | next_day +7.1 vs -1.7 (t +0.51)
2015- quad witching (index rebalance)  n 47: day -36.6 vs +8.1 (t -2.92) | next_on +6.3 vs -0.1 (t +0.61) | next_day +14.3 vs +4.7 (t +0.63)
2015- monthly opex Friday              n 94: day +2.7 vs +8.1 (t -0.49) | next_on -4.4 vs -0.1 (t -0.42) | next_day +0.6 vs +4.7 (t -0.33)


**Interpretation.** No reversal in either era (next open and next day, |t| ≤ 0.6). Quad-witching days themselves
were flat in 2004–14 (+10bp, t 0.7) and weak since 2015 (−37bp, t −2.9). That is a single-era pattern with no
in-sample support, and not a pressure-reversal mechanism.

**Verdict: fails the bar.**